# Applio One Click Gradio

กดรันเซลล์เดียวด้านล่าง แล้วรอจนขึ้นลิงก์ Gradio

In [ ]:
# @title กดรันครั้งเดียวเพื่อเปิด Applio Gradio
# @markdown ถ้า Colab ขอสิทธิ์ Google Drive ให้กดยืนยัน 1 ครั้ง แล้วปล่อยให้รันต่อ
mount_drive = True # @param {type:"boolean"}
sync_models_with_drive = True # @param {type:"boolean"}
sharing_method = "gradio" # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = "" # @param {type:"string"}
tensorboard = False # @param {type:"boolean"}

from IPython.display import clear_output
from pathlib import Path
import os

repo_url = "https://github.com/IAHispano/Applio/"
repo_name = "Applio"
repo_path = Path('/content') / repo_name
logs_path = repo_path / 'logs'
backups_path = Path('/content/drive/MyDrive/ApplioBackup')

if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')

%cd /content
!git config --global advice.detachedHead false
if repo_path.exists():
    %cd /content/Applio
    !git pull
else:
    !git clone {repo_url}
    %cd /content/Applio

clear_output()
print('กำลังติดตั้งของจำเป็น รอสักครู่...')
!apt update -y
!apt install -y portaudio19-dev
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
!uv pip install -q ngrok jupyter-ui-poll
!npm install -g -q localtunnel > /dev/null 2>&1
!python core.py prerequisites --models --pretraineds-hifigan

if mount_drive and sync_models_with_drive and Path('/content/drive').is_mount():
    print('กำลังเชื่อมโฟลเดอร์โมเดลกับ Google Drive...')
    backups_path.mkdir(parents=True, exist_ok=True)
    logs_path.mkdir(parents=True, exist_ok=True)
    temp_names = {'mute', 'reference', 'zips', 'mute_spin', 'mute_spin-v2'}
    for item in list(logs_path.iterdir()):
        if item.name not in temp_names and item.is_dir() and not item.is_symlink():
            target = backups_path / item.name
            if not target.exists():
                !mv "{item}" "{target}"
    for item in list(backups_path.iterdir()):
        link = logs_path / item.name
        if item.name not in temp_names and not link.exists():
            !ln -s "{item}" "{link}"

for name in ['mute', 'reference', 'zips', 'mute_spin', 'mute_spin-v2']:
    path = logs_path / name
    if path.exists() and (path.is_symlink() or not path.is_dir()):
        !rm -rf "{path}"
    path.mkdir(parents=True, exist_ok=True)

clear_output()
print('ติดตั้งเสร็จแล้ว กำลังเปิด Gradio...')

if tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir logs --bind_all

if sharing_method == 'gradio':
    !python app.py --listen --share --client
elif sharing_method == 'localtunnel':
    !echo Password IP: $(curl --silent https://ipv4.icanhazip.com)
    !lt --port 6969 & python app.py --listen --client
elif sharing_method == 'ngrok':
    import ngrok
    ngrok.kill()
    listener = await ngrok.forward(6969, authtoken=ngrok_token)
    print(f'Ngrok URL: {listener.url()}')
    !python app.py --listen --client
